# 04 Build Ablation Datasets

Build YOLO dataset folders for A/B/C/D ablation experiments from the original split and train-only offline augmentation.


## Purpose

Notebook 04 creates dataset folders and YAML files only. It does not train models, run online augmentation, generate new offline augmentations, or modify input/original split data.

Validation and test splits are copied identically into all experiments. Only the training set changes.


## Experiment Design

- `exp_A_original_only`: original train only, online augmentation OFF later.
- `exp_B_online_aug`: original train only, online augmentation ON later.
- `exp_C_offline_aug`: original train plus offline augmented train, online augmentation OFF/minimal later.
- `exp_D_full_pipeline`: original train plus offline augmented train, online augmentation ON later.

Online augmentation is not encoded into dataset YAMLs; it will be passed as training arguments in later notebooks.


## 1. Setup

Find the `sign-detection` root, add it to `sys.path`, and import the experiment dataset builder.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Notebook execution can start from the repository root or from notebooks/.
# This helper walks upward until it finds the sign-detection project layout.
def find_project_root(start: Path) -> Path:
    """Return the sign-detection project root from a notebook working directory."""
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "dataset_config.yaml").exists() and (
            candidate / "src"
        ).exists():
            return candidate
        sign_child = candidate / "sign-detection"
        if (sign_child / "configs" / "dataset_config.yaml").exists() and (
            sign_child / "src"
        ).exists():
            return sign_child
    raise RuntimeError("Could not find sign-detection project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())

# Add project root so reusable src/ modules import cleanly in Jupyter.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Keep this notebook as orchestration only. Copy/YAML/report logic lives in src/dataset/.
from src.dataset.build_experiment_sets import (
    build_ablation_datasets,
    collect_yolo_pairs,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Github\smart-factory-safety-monitoring-system\sign-detection


## 2. Load Configuration

Class names and dataset paths come from config. All paths are resolved relative to `sign-detection`.


In [2]:
# Load the fixed class mapping and dataset paths from config.
# This keeps generated YAMLs and experiment folders aligned with the module contract.
with (PROJECT_ROOT / "configs" / "class_names.yaml").open(
    "r", encoding="utf-8"
) as file:
    class_config = yaml.safe_load(file)
with (PROJECT_ROOT / "configs" / "dataset_config.yaml").open(
    "r", encoding="utf-8"
) as file:
    dataset_config = yaml.safe_load(file)

# Convert YAML class keys to int so the emitted YOLO YAML has numeric class IDs.
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}
paths = dataset_config["paths"]

# Notebook 04 reads original split and offline augmented train outputs.
# It writes only generated experiment datasets, reports, and dataset YAML files.
splits_original_dir = PROJECT_ROOT / paths["splits_original"]
augmented_train_dir = PROJECT_ROOT / paths["augmented_train"]
experiments_dir = PROJECT_ROOT / paths["experiments"]
reports_dir = PROJECT_ROOT / paths.get("reports_experiments", "reports/experiments")
output_yaml_dir = PROJECT_ROOT

# Reports are generated artifacts and can be created safely.
reports_dir.mkdir(parents=True, exist_ok=True)

print("Original split root:", splits_original_dir)
print("Augmented train root:", augmented_train_dir)
print("Experiments output root:", experiments_dir)
print("Experiment reports:", reports_dir)
print("Classes:", class_names)


Original split root: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\splits_original
Augmented train root: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\augmented_train
Experiments output root: C:\Github\smart-factory-safety-monitoring-system\sign-detection\data\generated\experiments
Experiment reports: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\experiments
Classes: {0: 'M014_Helmet', 1: 'M015_Vest', 2: 'P004_NoThoroughfare', 3: 'W011_Slippery'}


## 3. Check Required Inputs

The original train/val/test split is required. Offline augmented data is optional for running A/B, but C/D will warn if no augmented samples are available.


In [3]:
# Original train/val/test splits are required. They should come from Notebook 02.
required_original_splits = {
    "train": (
        splits_original_dir / "train" / "images",
        splits_original_dir / "train" / "labels",
    ),
    "val": (
        splits_original_dir / "val" / "images",
        splits_original_dir / "val" / "labels",
    ),
    "test": (
        splits_original_dir / "test" / "images",
        splits_original_dir / "test" / "labels",
    ),
}

input_check_rows = []
for split, (images_dir, labels_dir) in required_original_splits.items():
    # Collect same-stem image/label pairs without modifying the original split folders.
    pairs = collect_yolo_pairs(images_dir, labels_dir)
    input_check_rows.append(
        {
            "source": f"original_{split}",
            "images_dir": str(images_dir),
            "labels_dir": str(labels_dir),
            "num_pairs": len(pairs),
            "required": True,
        }
    )
    if not pairs:
        raise ValueError(
            f"Missing original {split} image-label pairs. Run Notebook 02 first."
        )

# Offline augmented data is optional for A/B, but C/D need it to become true offline/full-pipeline experiments.
# Missing augmented folders will become warnings rather than blocking A/B dataset creation.
for aug_type in ["geometric", "photometric", "weather_quality", "synthetic_placement"]:
    images_dir = augmented_train_dir / aug_type / "images"
    labels_dir = augmented_train_dir / aug_type / "labels"
    pairs = collect_yolo_pairs(images_dir, labels_dir)
    input_check_rows.append(
        {
            "source": f"augmented_{aug_type}",
            "images_dir": str(images_dir),
            "labels_dir": str(labels_dir),
            "num_pairs": len(pairs),
            "required": False,
        }
    )

# This preview makes it obvious whether Notebook 03 produced usable offline augmented samples.
input_check_df = pd.DataFrame(input_check_rows)
display(input_check_df)


,source,images_dir,labels_dir,num_pairs,required
0,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,332,True
1,original_val,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,96,True
2,original_test,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,47,True
3,augmented_geometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,100,False
4,augmented_photometric,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,83,False
5,augmented_weather_quality,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,66,False
6,augmented_synthetic_placement,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,0,False


## 4. Review Intended Experiment Design

This table makes the ablation contract explicit before files are copied.


In [4]:
# This table is a human-readable contract for what Notebook 04 will build.
# Online augmentation is not applied here; it is only recorded as a later training setting.
experiment_design_df = pd.DataFrame(
    [
        {
            "experiment": "exp_A_original_only",
            "train_data": "original train only",
            "val_data": "fixed original val",
            "test_data": "fixed original test",
            "online_augmentation_later": "OFF",
        },
        {
            "experiment": "exp_B_online_aug",
            "train_data": "original train only",
            "val_data": "fixed original val",
            "test_data": "fixed original test",
            "online_augmentation_later": "ON",
        },
        {
            "experiment": "exp_C_offline_aug",
            "train_data": "original train + offline augmented train",
            "val_data": "fixed original val",
            "test_data": "fixed original test",
            "online_augmentation_later": "OFF/minimal",
        },
        {
            "experiment": "exp_D_full_pipeline",
            "train_data": "original train + offline augmented train",
            "val_data": "fixed original val",
            "test_data": "fixed original test",
            "online_augmentation_later": "ON",
        },
    ]
)

# Safety switch: keep False for normal use. Set True only to intentionally rebuild experiment folders.
OVERWRITE_EXPERIMENTS = False

print("Overwrite existing experiment folders:", OVERWRITE_EXPERIMENTS)
display(experiment_design_df)


Overwrite existing experiment folders: False


,experiment,train_data,val_data,test_data,online_augmentation_later
0,exp_A_original_only,original train only,fixed original val,fixed original test,OFF
1,exp_B_online_aug,original train only,fixed original val,fixed original test,ON
2,exp_C_offline_aug,original train + offline augmented train,fixed original val,fixed original test,OFF/minimal
3,exp_D_full_pipeline,original train + offline augmented train,fixed original val,fixed original test,ON


## 5. Build Ablation Dataset Folders

The builder copies files into `data/generated/experiments/` and writes four Ultralytics dataset YAML files. Original splits and augmented source folders remain untouched.


In [5]:
# Build all four experiment datasets in one pass.
# The helper copies files only; original splits and augmented_train folders stay untouched.
copy_report_df, summary_df, warnings_df = build_ablation_datasets(
    splits_original_dir=splits_original_dir,
    augmented_train_dir=augmented_train_dir,
    experiments_dir=experiments_dir,
    class_names=class_names,
    output_yaml_dir=output_yaml_dir,
    overwrite=OVERWRITE_EXPERIMENTS,
)

print("Ablation dataset folders built.")
print(f"Copied rows: {len(copy_report_df)}")


Ablation dataset folders built.
Copied rows: 2398


## 6. Save Reports

Reports are generated artifacts under `reports/experiments/`. They make file copies, counts, and warnings auditable.


In [6]:
# Save the audit trail: every copied pair, compact counts, and integrity warnings.
copy_report_path = reports_dir / "ablation_dataset_report.csv"
summary_path = reports_dir / "ablation_dataset_summary.csv"
warnings_path = reports_dir / "ablation_integrity_warnings.csv"

copy_report_df.to_csv(copy_report_path, index=False)
summary_df.to_csv(summary_path, index=False)
warnings_df.to_csv(warnings_path, index=False)

print(f"Saved copy report: {copy_report_path}")
print(f"Saved summary: {summary_path}")
print(f"Saved warnings: {warnings_path}")


Saved copy report: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\experiments\ablation_dataset_report.csv
Saved summary: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\experiments\ablation_dataset_summary.csv
Saved warnings: C:\Github\smart-factory-safety-monitoring-system\sign-detection\reports\experiments\ablation_integrity_warnings.csv


## 7. Confirm YAML Outputs

Each YAML points Ultralytics at one generated experiment dataset. Online augmentation will be configured later during training, not here.


In [7]:
# Confirm all experiment YAML files are present and inspect their Ultralytics paths.
# These YAML files do not contain online augmentation settings; training notebooks will pass those args.
yaml_paths = sorted(PROJECT_ROOT.glob("data_exp_*.yaml"))
for yaml_path in yaml_paths:
    print(yaml_path.name)
    with yaml_path.open("r", encoding="utf-8") as file:
        display(yaml.safe_load(file))


data_exp_A_original_only.yaml


{'path': 'data/generated/experiments/exp_A_original_only',
 'train': 'train/images',
 'val': 'val/images',
 'test': 'test/images',
 'nc': 4,
 'names': {0: 'M014_Helmet',
  1: 'M015_Vest',
  2: 'P004_NoThoroughfare',
  3: 'W011_Slippery'}}

data_exp_B_online_aug.yaml


{'path': 'data/generated/experiments/exp_B_online_aug',
 'train': 'train/images',
 'val': 'val/images',
 'test': 'test/images',
 'nc': 4,
 'names': {0: 'M014_Helmet',
  1: 'M015_Vest',
  2: 'P004_NoThoroughfare',
  3: 'W011_Slippery'}}

data_exp_C_offline_aug.yaml


{'path': 'data/generated/experiments/exp_C_offline_aug',
 'train': 'train/images',
 'val': 'val/images',
 'test': 'test/images',
 'nc': 4,
 'names': {0: 'M014_Helmet',
  1: 'M015_Vest',
  2: 'P004_NoThoroughfare',
  3: 'W011_Slippery'}}

data_exp_D_full_pipeline.yaml


{'path': 'data/generated/experiments/exp_D_full_pipeline',
 'train': 'train/images',
 'val': 'val/images',
 'test': 'test/images',
 'nc': 4,
 'names': {0: 'M014_Helmet',
  1: 'M015_Vest',
  2: 'P004_NoThoroughfare',
  3: 'W011_Slippery'}}

## 8. Review Dataset Summary

Check that A/B have original-only train counts and C/D include offline augmented training data. Validation and test counts should match across all experiments.


In [8]:
# Summary should show A/B with original-only train counts and C/D with larger offline-augmented train counts.
display(summary_df)

train_counts = summary_df[summary_df["split"] == "train"][
    ["experiment", "num_images", "num_labels", "total_objects", "num_no_sign_images"]
]
print("Train split counts by experiment")
display(train_counts)

# Validation and test counts should be identical across all four experiments.
val_test_counts = summary_df[summary_df["split"].isin(["val", "test"])][
    ["experiment", "split", "num_images", "num_labels", "total_objects"]
]
print("Fixed val/test counts")
display(val_test_counts)


,experiment,split,num_images,num_labels,num_no_sign_images,total_objects,notes,num_M014_Helmet,num_M015_Vest,num_P004_NoThoroughfare,num_W011_Slippery
0,exp_A_original_only,train,332,332,21,366,online augmentation handled later during training,92,90,94,90
1,exp_A_original_only,val,96,96,6,100,online augmentation handled later during training,25,25,24,26
2,exp_A_original_only,test,47,47,3,48,online augmentation handled later during training,12,11,12,13
3,exp_B_online_aug,train,332,332,21,366,online augmentation handled later during training,92,90,94,90
4,exp_B_online_aug,val,96,96,6,100,online augmentation handled later during training,25,25,24,26
5,exp_B_online_aug,test,47,47,3,48,online augmentation handled later during training,12,11,12,13
6,exp_C_offline_aug,train,581,581,39,612,online augmentation handled later during training,161,149,147,155
7,exp_C_offline_aug,val,96,96,6,100,online augmentation handled later during training,25,25,24,26
8,exp_C_offline_aug,test,47,47,3,48,online augmentation handled later during training,12,11,12,13
9,exp_D_full_pipeline,train,581,581,39,612,online augmentation handled later during training,161,149,147,155


Train split counts by experiment


,experiment,num_images,num_labels,total_objects,num_no_sign_images
0,exp_A_original_only,332,332,366,21
3,exp_B_online_aug,332,332,366,21
6,exp_C_offline_aug,581,581,612,39
9,exp_D_full_pipeline,581,581,612,39


Fixed val/test counts


,experiment,split,num_images,num_labels,total_objects
1,exp_A_original_only,val,96,96,100
2,exp_A_original_only,test,47,47,48
4,exp_B_online_aug,val,96,96,100
5,exp_B_online_aug,test,47,47,48
7,exp_C_offline_aug,val,96,96,100
8,exp_C_offline_aug,test,47,47,48
10,exp_D_full_pipeline,val,96,96,100
11,exp_D_full_pipeline,test,47,47,48


## 9. Review Copy Report and Warnings

Filename conflicts are resolved with deterministic suffixes and recorded. Warnings are review prompts unless a copy failed.


In [9]:
# The copy report is the first place to inspect if a file was skipped, failed, or renamed with a conflict suffix.
print("Copy report preview")
display(copy_report_df.head(20))

# Warnings are review prompts. They are especially useful for missing augmentation data or val/test mismatches.
if warnings_df.empty:
    print("No integrity warnings generated.")
else:
    display(warnings_df)


Copy report preview


,experiment,split,source_type,original_image_path,original_label_path,copied_image_path,copied_label_path,status,notes
0,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
1,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
2,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
3,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
4,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
5,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
6,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
7,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
8,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
9,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,


No integrity warnings generated.


## 10. Notebook 05 Checklist

Before moving to Notebook 05:

- Confirm four experiment folders exist under `data/generated/experiments/`.
- Confirm `data_exp_A_original_only.yaml` through `data_exp_D_full_pipeline.yaml` exist.
- Confirm validation and test counts match across all experiments.
- Review any filename conflict or missing-data warnings.
- Remember that online augmentation ON/OFF is handled later by training arguments, not by these dataset folders.
